<a href="https://colab.research.google.com/github/JFSS20000/07MIAR04/blob/Actividad_Articulo_C1/C2_ResUnet_JOSE_FERNANDO_SARMIENTO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# Instalar el modelo del repositorio original

!python -m pip install git+https://github.com/JanPalasek/resunet-tensorflow

  Cloning https://github.com/JanPalasek/resunet-tensorflow to /tmp/pip-req-build-509g8_lt
  Running command git clone --filter=blob:none --quiet https://github.com/JanPalasek/resunet-tensorflow /tmp/pip-req-build-509g8_lt
  Resolved https://github.com/JanPalasek/resunet-tensorflow to commit 9f080caba27441a16697818cb7064839c40223e5
  Preparing metadata (setup.py) ... done
  Created wheel for resunet: filename=resunet-1.1-py3-none-any.whl size=2822 sha256=c2149e7d310f0c0350af464f52e523e2573fd721af4e55f89be452056c48feb2
  Stored in directory: /tmp/pip-ephem-wheel-cache-egy5_438/wheels/c0/2d/6e/f4e737af24afd540201857bdf3ac8c3149e36c5fba075c4487
Successfully built resunet


In [5]:
# Entrenar el modelo en este proyecto

from resunet import ResUNet
import tensorflow as tf
import tensorflow_datasets as tfds

# 1. DESCARGAR Y PREPARAR EL DATASET (Oxford-IIIT Pet)
def normalize(input_image, input_mask):
    input_image = tf.cast(input_image, tf.float32) / 255.0
    # El dataset tiene etiquetas 1, 2, 3. Restamos 1 para tener 0, 1, 2
    input_mask -= 1
    # Para este ejemplo de 2 clases, convertimos a binario: Mascota (0) vs Resto (1)
    input_mask = tf.where(input_mask > 0, 1, 0)
    # Convertir máscara a One-Hot para usar categorical_crossentropy
    input_mask = tf.one_hot(input_mask, depth=2)
    return input_image, input_mask

dataset, info = tfds.load('oxford_iiit_pet:3.*.*', with_info=True)

def load_image(datapoint):
    input_image = tf.image.resize(datapoint['image'], (128, 128))
    input_mask = tf.image.resize(datapoint['segmentation_mask'], (128, 128))
    input_image, input_mask = normalize(input_image, input_mask)
    return input_image, input_mask

# Crear pipelines de entrenamiento y validación
TRAIN_LENGTH = info.splits['train'].num_examples
BATCH_SIZE = 32
BUFFER_SIZE = 1000

train_batches = (
    dataset['train']
    .map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
    .cache()
    .shuffle(BUFFER_SIZE)
    .batch(BATCH_SIZE)
    .prefetch(buffer_size=tf.data.AUTOTUNE)
)

val_batches = (
    dataset['test']
    .batch(BATCH_SIZE)
    .map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
)

# 2. CONFIGURAR EL MODELO RESUNET
#### create model for inputs of sizes (128, 128, 1) for semantic segmentation into 2 classes
#### architecture will have 16 filters in the root and the depth of 3 blocks
#### model = ResUNet(input_shape=(128, 128, 1), classes=2, filters_root=16, depth=3)

# Cambiamos input_shape a (128, 128, 3) porque el dataset es a color (RGB)
model = ResUNet(input_shape=(128, 128, 3), classes=2, filters_root=16, depth=3)

# compile the model
# categorical crossentropy is the preferred loss function
##model.compile(loss="categorical_crossentropy", optimizer="adam",
##                  metrics=["categorical_accuracy", "some other metrics"])
model.compile(loss="categorical_crossentropy", optimizer="adam",
                  metrics=["accuracy"] )

# 3. ENTRENAMIENTO
# use model.fit, model.evalute as with any other tf2 model
#model.fit(x=x, y=y, validation_data=validation_dataset, epochs=args.epochs, batch_size=args.batch_size)
model.fit(
    train_batches,
    epochs=10,
    validation_data=val_batches
)

# 4. MOSTRAR RESUMEN DE PARÁMETROS
model.summary()



AssertionError: Failed to construct dataset "oxford_iiit_pet", builder_kwargs "{'version': '3.*.*', 'data_dir': None}": Dataset oxford_iiit_pet cannot be loaded at version 3.*.*, only: 4.0.0.